# Week 15 -- gates

Pre-registration: `W15_PREREG.md`. Nothing here fits a model. G15.1 and G15.2
can still change the week's design; G15.4 decides which folds are resolvable.


In [1]:
# --- ADAPTER CELL -------------------------------------------------------
# Repo coupling lives here and nowhere else.  Prints the archive's real
# contents and asserts before anything downstream runs.  Four wrong assumptions
# were caught this way in Week 13, each in one round trip.

import sys, os
import numpy as np
sys.path.insert(0, os.path.join('..', 'src'))

from splits import (groups_within_levels, _straddling_groups,
                    leave_one_level_out, group_census, split_report)
import constants

ARCHIVE = '../data/qpit_dataset_w10.npz'
z = np.load(ARCHIVE, allow_pickle=True)

print(f"{ARCHIVE}: {len(z.files)} keys")
for k in sorted(z.files):
    a = z[k]
    print(f"  {k:<22} {str(a.shape):<14} {a.dtype}")

REQUIRED = ['group_id', 'family_id', 'energy_id', 'C', 'd_km', 'at_floor',
            'family_tags', 'floor_km', 'gap', 'family_max_gap_km']
missing = [k for k in REQUIRED if k not in z.files]
assert not missing, f"archive is missing required columns: {missing}"

gid, fid, eid = z['group_id'], z['family_id'], z['energy_id']
C, d_km, at_floor = z['C'], z['d_km'], z['at_floor']
tags = z['family_tags']

# floor_km is carried by the frozen dataset itself; read it from the archive
# and cross-check against constants.py.  That is the drift check, not shadowing.
FLOOR_KM = float(z['floor_km'])
assert abs(FLOOR_KM - constants.FLOOR_KM) < 1e-12, (
    f"archive floor_km {FLOOR_KM} disagrees with constants.py "
    f"{constants.FLOOR_KM}")

n = len(gid)
assert all(z[k].shape[0] == n for k in ('family_id', 'energy_id', 'C', 'd_km')), \
    "per-row columns disagree on length"
print(f"\nrows={n}  measured={int((~at_floor).sum())}  "
      f"censored={int(at_floor.sum())}  FLOOR_KM={FLOOR_KM}")


../data/qpit_dataset_w10.npz: 31 keys
  A_target_km            ()             float64
  C                      (20400,)       float64
  alpha                  (20400,)       float64
  amplitude_km           (20400,)       float64
  at_floor               (20400,)       bool
  d_drawn_km             (20400,)       float64
  d_km                   (20400,)       float64
  energies               (3,)           float64
  energy_id              (20400,)       int64
  family                 (20400,)       <U2
  family_id              (20400,)       int64
  family_max_gap_km      (6,)           float64
  family_sizes           (6,)           int64
  family_tags            (6,)           <U5
  floor_km               ()             float64
  gap                    (20400,)       int64
  group_id               (20400,)       int64
  jacobi                 (20400,)       float64
  label_B                (20400,)       bool
  label_B_km             ()             float64
  mu                     (

## G15.1 -- group atomicity

No `group_id` may straddle a level boundary on either key. A straddler between two *non*-held-out levels passes `groups_disjoint` on every seed of that fold while breaking a different one, so this is checked upfront, seed-independently, and not inferred from a split.

In [2]:
G15_1 = {}
for name, key in (('family_id', fid), ('energy_id', eid)):
    bad = _straddling_groups(gid, key)
    G15_1[name] = dict(atomic=bool(groups_within_levels(gid, key)),
                       n_straddling=int(len(bad)),
                       examples=bad[:5].tolist())
    print(f"  {name:<10} atomic={G15_1[name]['atomic']}  "
          f"straddling={G15_1[name]['n_straddling']}")

assert all(v['atomic'] for v in G15_1.values()), \
    "G15.1 FAILED -- a level holdout would leak; stop and redesign the split"
print("\nG15.1 PASS")


  family_id  atomic=True  straddling=0
  energy_id  atomic=True  straddling=0

G15.1 PASS


## G15.2 -- energy ordering by C

`C` is one-to-one with `energy_id` but not guaranteed monotone in it. The middle energy is identified by sorted `C` value. P15.3's interpolation/extrapolation reading is void if this is skipped.

In [3]:
assert all(len(np.unique(C[eid == e])) == 1 for e in np.unique(eid)), \
    "C is not constant within energy_id; the conditioning cells are not what "\
    "the pre-registration assumes"

order = sorted((float(C[eid == e][0]), int(e)) for e in np.unique(eid))
E_a, E_b, E_c = (e for _, e in order)
ROLES = {E_a: 'E_a low  (extrapolation)',
         E_b: 'E_b MID  (interpolation)',
         E_c: 'E_c high (extrapolation)'}
for c, e in order:
    print(f"  C={c:.6f}  energy_id={e}   {ROLES[e]}")

MONOTONE = [e for _, e in order] == sorted(e for _, e in order)
print(f"\nenergy_id monotone in C: {MONOTONE}")
print("G15.2 PASS -- folds are keyed by C rank, not by energy_id")


  C=3.029398  energy_id=0   E_a low  (extrapolation)
  C=3.064703  energy_id=1   E_b MID  (interpolation)
  C=3.130607  energy_id=2   E_c high (extrapolation)

energy_id monotone in C: True
G15.2 PASS -- folds are keyed by C rank, not by energy_id


## Cell census

Six conditioning cells (family x energy). Group-size structure drives the divergence between group fractions and row fractions.

In [4]:
cells = fid * 3 + eid
u, counts = np.unique(cells, return_counts=True)
for i, t in enumerate(tags):
    print(f"  {t}: {counts[i]} rows")
cen = group_census(gid)
print(f"\ngroups={cen['n_groups']}  median size={cen['size_median']}  "
      f"max={cen['size_max']}  singletons={cen['n_singleton']}")


  L1_E0: 3400 rows
  L1_E1: 3400 rows
  L1_E2: 3400 rows
  L2_E0: 3400 rows
  L2_E1: 3400 rows
  L2_E2: 3400 rows

groups=6564  median size=2.0  max=24  singletons=1762


## G15.4 -- far-field test n per fold

Transfer is evaluated in the far field only; the near field carries no detectable information in distribution, so a near-field transfer measurement would measure the transfer of nothing. A fold with far-field test n below 500 is reported unresolved by construction.

In [5]:
# The dead-zone boundary is NOT carried by the archive, so it comes from
# constants.py.  `DEAD_ZONE_KM` was a guessed name and does not exist; the
# candidates below are also guesses, and if none hits, this prints the real
# float attributes so the right name can be filled in once.
#
# The name is not the guard.  Re-deriving the boundary as the midpoint of the
# log10 span gives 0.124302 km and 8909 far-field rows -- one row off Week 13's
# 8908 -- so a plausible-looking derivation silently moves the window and makes
# Week 15 incomparable to the Week 13 MI result.  The assertion on the COUNT is
# the guard, because that is the condition that actually matters.

# RESOLVED: there is no dead-zone boundary in constants.py.  Its float
# attributes are FLOOR_KM, HALO29_C, HALO29_PERIOD, LU, LU_KM,
# MU_EARTH_MOON -- so 0.12462 was derived in Week 13 and never promoted.
#
# Prefer the Week 13 source if it is importable; otherwise declare it here and
# say so.  Neither path is the guard: the assertion on the far-field COUNT is.
# Re-deriving the boundary as the midpoint of the log10 span gives 0.124302 km
# and 8909 rows -- one row off -- so a principled-looking derivation silently
# moves the window and makes Week 15 incomparable to the Week 13 MI result.
#
# The count is reproduced by any boundary in (0.124390749, 0.124654774] km, a
# band 0.21% wide, so this needs four decimals: 0.1246 works, 0.125 does not.
# TODO promote to src/constants.py as DEAD_ZONE_KM = 0.12462 and delete the
# fallback below.

DEAD_ZONE_KM_W13 = 0.12462
BOUNDARY_KM, _src = None, None
for _mod, _fn in (('mi', 'make_windows'), ('metrics', 'make_windows'),
                  ('windows', 'make_windows')):
    try:
        _m = __import__(_mod)
        if hasattr(_m, _fn):
            BOUNDARY_KM, _src = None, f"{_mod}.{_fn}"
            break
    except ImportError:
        continue
if _src is not None:
    print(f"NOTE: {_src} exists -- the boundary should come from it, not from "
          f"the literal below.  Wire it up and re-run.")
if BOUNDARY_KM is None:
    BOUNDARY_KM = DEAD_ZONE_KM_W13
    _src = "Week-15-local literal (not in constants.py)"
print(f"dead-zone boundary = {BOUNDARY_KM} km   source: {_src}")

far = (~at_floor) & (d_km > BOUNDARY_KM)
N_FAR_WK13 = 8908
print(f"far-field measured rows overall: {int(far.sum())}  "
      f"(Week 13 matched the MI windows at n = {N_FAR_WK13})")
assert int(far.sum()) == N_FAR_WK13, (
    f"far-field window gives {int(far.sum())} rows against Week 13's "
    f"{N_FAR_WK13}; the window definition has drifted and Week 15 far-field "
    f"numbers are not comparable to the Week 13 MI result")

FOLDS = [('T1  train L1 -> test L2', fid, 1),
         ('T2  train L2 -> test L1', fid, 0),
         ('T3  hold out E_a', eid, E_a),
         ('T4  hold out E_b', eid, E_b),
         ('T5  hold out E_c', eid, E_c)]

G15_4 = {}
for label, key, lv in FOLDS:
    m, meta = leave_one_level_out(key, lv, gid, frac_test=0.2,
                                  frac_val=0.1, seed=0)
    n_far = int((m['test'] & far).sum())
    G15_4[label] = dict(meta=meta, far_test_n=n_far,
                        resolvable=bool(n_far >= 500))
    print(f"  {label:<24} test={meta['test_rows']:>5}  far={n_far:>5}  "
          f"train={meta['train_rows']:>6}  excl={meta['excluded_rows']:>5}  "
          f"resolvable={n_far >= 500}")

# split_report per fold (G15.5).  A level holdout shows a large KS statistic by
# construction; the number quantifies how different the held-out region is.
print()
for label, key, lv in FOLDS:
    m, _ = leave_one_level_out(key, lv, gid, frac_test=0.2, frac_val=0.1, seed=0)
    r = split_report(np.log10(np.maximum(d_km, FLOOR_KM)), m, group_id=gid)
    print(f"  {label:<24} KS(train,test)={r['ks_stat_train_test']:.4f}")


NOTE: mi.make_windows exists -- the boundary should come from it, not from the literal below.  Wire it up and re-run.
dead-zone boundary = 0.12462 km   source: Week-15-local literal (not in constants.py)
far-field measured rows overall: 8908  (Week 13 matched the MI windows at n = 8908)
  T1  train L1 -> test L2  test= 2121  far=  923  train=  9216  excl= 8079  resolvable=True
  T2  train L2 -> test L1  test= 1970  far=  849  train=  9177  excl= 8230  resolvable=True
  T3  hold out E_a         test= 1385  far=  625  train= 12280  excl= 5415  resolvable=True
  T4  hold out E_b         test= 1344  far=  574  train= 12243  excl= 5456  resolvable=True
  T5  hold out E_c         test= 1360  far=  569  train= 12277  excl= 5440  resolvable=True

  T1  train L1 -> test L2  KS(train,test)=0.0200
  T2  train L2 -> test L1  KS(train,test)=0.0189
  T3  hold out E_a         KS(train,test)=0.0246
  T4  hold out E_b         KS(train,test)=0.0191
  T5  hold out E_c         KS(train,test)=0.0289


## Not run here

`nearest_point_overlap` is declared not applicable to a level holdout (W15_PREREG.md §6a) and is not called. The recovered coordinates it documents as its inputs (`alpha_found`, `theta_found`) are absent from this archive in any case.